<a href="https://colab.research.google.com/github/jhenningsen/Equity_Analysis/blob/main/Data_Download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Volatility Breakout Strategy & Daily Scanner

This notebook implements a technical analysis scanner designed to identify **Volatility Breakouts**. The strategy focuses on identifying stocks that experience significant price movements (up or down) relative to their historical volatility.

**Key Features:**
*   **Dynamic Signal Detection:** Identifies trigger days where the absolute daily return exceeds a multiple of the historical rolling volatility (configurable via `TARGET_MULTIPLIER`). Signals are detected for both 'up' and 'down' movements.
*   **Cooldown Mechanism:** Prevents redundant signals by enforcing a 'cool-off' period (`COOLDOWN_DAYS`) after a trigger is detected for a specific ticker in each direction.
*   **Ticker Loading:** Loads ticker symbols from a Google Drive file or falls back to a predefined list.
*   **Historical Data Fetching:** Downloads historical stock data using `yfinance`.
*   **Forward Performance Tracking:** Automatically calculates forward price changes (1-day through 5-day) *per share* following each breakout.
*   **Parameterized Retracement Analysis:** Analyzes the probability of price retracement, calculated as a percentage of the initial signal day's return, over the subsequent 5 days, customizable by a `TARGET_MULTIPLIER`.

In [33]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

In [34]:
# Clear all DataFrames from memory
import gc

# Get a list of all variables in the global namespace
all_vars = list(globals().keys())

# Identify and delete pandas DataFrames
for var_name in all_vars:
    if isinstance(globals()[var_name], pd.DataFrame):
        del globals()[var_name]
        print(f"Deleted DataFrame: {var_name}")

# Run garbage collector to free up memory
gc.collect()

print("All DataFrames cleared from memory.")

Deleted DataFrame: tickers_df
Deleted DataFrame: data
Deleted DataFrame: data_long
All DataFrames cleared from memory.


In [35]:
# These are Google Drive file IDs. To get your own, right-click on the file in Google Drive, select 'Share', then 'Get link'. The ID is the part of the URL after 'id='.
OptionVolume_id = '1OGdLINK3zjlx6-lMq86SVq9TkbcglkeI'
OptionVolume = f'https://drive.google.com/uc?export=download&id={OptionVolume_id}'

OptionVolume200_id = '1gcwD510l4GFGNcKsbExR3GvKnDZwCHy4'
OptionVolume200 = f'https://drive.google.com/uc?export=download&id={OptionVolume200_id}'

### Loading Tickers from the File

In [36]:
try:
    # Read the CSV file into a DataFrame
    tickers_df = pd.read_csv(OptionVolume)

    # Attempt to directly use the 'Symbol' column
    try:
        tickers = tickers_df['Symbol'].tolist()
    except KeyError:
        # Fallback to the first column if 'Symbol' column is not found
        print(" 'Symbol' column not found, falling back to the first column.")
        tickers = tickers_df.iloc[:, 0].tolist()

    # Remove any potential NaN values or empty strings
    tickers = [t for t in tickers if isinstance(t, str) and t.strip() != '']

    print(f"Loaded {len(tickers)} tickers: {tickers[:10]}...") # Display first 10 tickers
except Exception as e:
    print(f"Error reading tickers file: {e}")
    tickers = [] # Fallback to an empty list if there's an error

if not tickers:
    print("No tickers loaded. Falling back to a predefined list for demonstration.")
    tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA'] # Example list

Loaded 100 tickers: ['SPY', 'TSLA', 'QQQ', 'NVDA', 'MSFT', 'MU', 'META', 'AMZN', 'MSTR', 'AAPL']...


In [37]:
start_date_str_input = '01/01/2020'
end_date_str_input = '08/08/2026'

# Convert date strings to datetime objects
start_date = pd.to_datetime(start_date_str_input)
end_date = pd.to_datetime(end_date_str_input)

print(f"Downloading data for {len(tickers)} tickers from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}...")
data = yf.download(tickers, start=start_date, end=end_date)
print("Download complete.")

/tmp/ipykernel_1673/4038562416.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(tickers, start=start_date, end=end_date)
[                       0%                       ]

[*********************100%***********************]  100 of 100 completed


Download complete.


### Fetching Historical Data from Yahoo Finance

The `yf.download()` function by default returns a DataFrame with a multi-level column index, where the first level indicates the data type (e.g., 'Adj Close', 'Open', 'Volume') and the second level indicates the ticker symbol. This can be viewed as a 'wide' format.

To get the data in a 'long' format, where each row represents a single observation for a given `(Date, Ticker)` pair, you can `stack()` the DataFrame and then reset the index. This will make it easier to work with individual ticker data.

In [38]:
# To transform the wide DataFrame into a 'long' format:
# First, sort the columns to ensure consistent stacking order (optional but good practice)
data_long = data.stack(level=1, future_stack=True).reset_index()

# Rename columns for clarity
data_long.rename(columns={'level_1': 'Ticker', 'level_0': 'Date'}, inplace=True)

# Display the first few rows of the transformed data
print("Downloaded data in 'long' format:")
display(data_long.head())

# Use the start_date and end_date from the data download step
# These variables (start_date, end_date) are already defined in the notebook context
# from cell e4a4cb07, so we don't need to redefine them here.

# Format dates for filename using the already established start_date and end_date
start_date_str = start_date.strftime('%Y-%m-%d')
end_date_str = end_date.strftime('%Y-%m-%d')

# Create filename with start and end dates
output_filename = f'yahoo_finance_data_{start_date_str}_{end_date_str}.parquet'

# Save the DataFrame to a Parquet file
data_long.to_parquet(output_filename, index=False)
print(f"Data saved to {output_filename}")

Downloaded data in 'long' format:


Price,Date,Ticker,Close,High,Low,Open,Volume
0,2020-01-02,AAOI,12.500000,12.530000,11.800000,12.130000,885000.0
1,2020-01-02,AAPL,72.333878,72.394086,71.091184,71.344054,135480400.0
2,2020-01-02,ADBE,334.429993,334.480011,329.170013,330.000000,1990100.0
3,2020-01-02,AMAT,58.626591,59.050739,58.155316,58.438080,6647900.0
4,2020-01-02,AMD,49.099998,49.250000,46.630001,46.860001,80331100.0


Data saved to yahoo_finance_data_2020-01-01_2026-08-08.parquet
